In [1]:
# モジュールのインポート
import os
import psutil
import time
import numpy as np
import pandas as pd
import csv
import optuna
from PIL import Image
from matplotlib import pylab as plt
import tifffile
import warnings
import geopandas as gpd
import random
import glob
from shapely.geometry import box
from rasterio.features import rasterize
from affine import Affine
from optuna.pruners import MedianPruner
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torchmetrics.classification import BinaryAUROC
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

In [2]:
# ===== 再現性確保のためのシード固定 =====
import os
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int = SEED):
    """random / numpy / torch (CPU・GPU) のシードを固定し、cuDNN を決定論的にする。"""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [3]:
dataset = pd.read_csv("../../../../くま発見3次メッシュデータセット_東北全県.csv")
dataset["kuma_view"] = 1*(dataset["kuma_view"]>0)

# ===== 実験2(持ち回り5fold): 共有5分割の割当(fold_group)を付与。割合は実験3の県別件数に一致 =====
dataset["fold_group"] = pd.read_csv("fold_assignment.csv")["fold_group"].values
assert dataset["fold_group"].notna().all(), "fold_group 欠損あり"


In [4]:
# --- 3次メッシュコード -> 緯度経度のバウンディングボックス（WGS84） ---
def mesh3_bbox(code: str):
    """
    8桁の3次メッシュコード -> (lon_min, lat_min, lon_max, lat_max) in degrees (EPSG:4326)
    """
    code = str(code)
    if len(code) != 8 or not code.isdigit():
        raise ValueError(f"Invalid 3rd-mesh code: {code} (must be 8 digits)")
    # 第1次メッシュ
    a = int(code[0:2])      # 緯度 40' 単位の番号 = floor(lat * 1.5)
    b = int(code[2:4])      # 経度 1° 単位の番号 = floor(lon - 100)
    # 第2次メッシュの添字
    p = int(code[4])        # 緯度方向 5' のブロック 0-7
    q = int(code[5])        # 経度方向 7.5' のブロック 0-7
    # 第3次メッシュの添字
    r = int(code[6])        # 緯度方向 30" のブロック 0-9
    s = int(code[7])        # 経度方向 45" のブロック 0-9

    # 単位を度に換算
    lat_base = a / 1.5                           # 40' = 2/3°
    lon_base = b + 100
    lat_min = lat_base + (p * (5/60)) + (r * (30/3600))
    lon_min = lon_base + (q * (7.5/60)) + (s * (45/3600))
    lat_max = lat_min + (30/3600)
    lon_max = lon_min + (45/3600)
    return (lon_min, lat_min, lon_max, lat_max)

# 指定した bbox を 256x256 のピクセルグリッドに
def bounds_to_transform(lon_min, lat_min, lon_max, lat_max, width=256, height=256):
    # 左上原点のアフィン（行列は行方向が北→南なので y ピクセルサイズは負）
    xres = (lon_max - lon_min) / width
    yres = (lat_min - lat_max) / height  # negative
    return Affine.translation(lon_min, lat_max) * Affine.scale(xres, yres)

# メッシュ毎 × カテゴリ毎に one-hot でラスタライズ
def build_mesh_tensor(mesh_codes, geojson_path, category_field, categories,
                      width=256, height=256, all_touched=True, dtype=np.uint8):
    """
    Returns: array with shape (num_mesh, height, width, num_categories)
    """
    # 読み込み（CRS は自動検出。なければ WGS84 とみなす）
    gdf = geojson_path
    if gdf.crs is None:
        # 必要に応じて EPSG:4326 を仮定（GeoJSONは通常WGS84）
        gdf.set_crs(epsg=4326, inplace=True)
    else:
        gdf = gdf.to_crs(epsg=4326)

    # カテゴリ → バンド index
    cat_to_idx = {c: i for i, c in enumerate(categories)}

    out = np.zeros((len(mesh_codes), len(categories), height, width), dtype=dtype)

    for mi, code in enumerate(mesh_codes):
        lon_min, lat_min, lon_max, lat_max = mesh3_bbox(code)
        bbox_poly = box(lon_min, lat_min, lon_max, lat_max)

        # 対象メッシュにかかるフィーチャだけへ絞る（高速化）
        sub = gdf[gdf.geometry.intersects(bbox_poly)]
        if sub.empty:
            continue

        transform = bounds_to_transform(lon_min, lat_min, lon_max, lat_max, width, height)

        # カテゴリごとに 0/1 バンドとして焼く（one-hot）
        for cat, idx in cat_to_idx.items():
            shp = sub[sub[category_field] == cat]
            if shp.empty:
                continue

            shapes = ((geom.intersection(bbox_poly), 1) for geom in shp.geometry if not geom.is_empty)

            band = rasterize(
                shapes=shapes,
                out_shape=(height, width),
                transform=transform,
                fill=0,
                all_touched=all_touched,
                dtype=dtype
            )
            out[mi,idx, :, :] = np.where(band > 0, 1, out[mi,idx, :, :])
    return out

In [5]:
gdf2020 = gpd.read_file("../../../../植生/現存植生図2024_2020_東北ブロック.geojson")
gdf2022 = gpd.read_file("../../../../植生/現存植生図2024_2022_東北ブロック.geojson")

In [6]:
len(set(gdf2022["植生区分"]))

6

In [7]:
set(gdf2022["植生区分"])

{'その他',
 'コケモモ－トウヒクラス域自然植生',
 'ブナクラス域代償植生',
 'ブナクラス域自然植生',
 '植林地・耕作地植生',
 '河辺・湿原・塩沼地・砂丘植生'}

In [8]:
len(set(gdf2020["植生区分"]))

9

In [9]:
set(gdf2020["植生区分"])

{'その他',
 'コケモモ－トウヒクラス域自然植生',
 'ブナクラス域代償植生',
 'ブナクラス域自然植生',
 'ヤブツバキクラス域代償植生',
 'ヤブツバキクラス域自然植生',
 '植林地・耕作地植生',
 '河辺・湿原・塩沼地・砂丘植生',
 '高山帯自然植生域'}

In [10]:
def makedata(indexlist, gdf2020, gdf2022):
    mesh_codes = dataset["meshcode"][indexlist]
    category_field = "凡例名"; categories = set(gdf2020["凡例名"])
    category_field = "植生自然度区分"; categories = set(gdf2020["植生自然度区分"])
    X_data4C_2020 = build_mesh_tensor(mesh_codes, gdf2020, category_field, categories)
    X_data4C_2022 = build_mesh_tensor(mesh_codes, gdf2022, category_field, categories)
    X_data4C_2020 = X_data4C_2020>0
    X_data4C_2022 = X_data4C_2022>0
    X_data4C = ((X_data4C_2022 + X_data4C_2020) > 0)
    del X_data4C_2022, X_data4C_2020
    geojson_path = gdf2020
    category_field = "凡例名"; categories = set(gdf2020["凡例名"])
    category_field = "植生区分"; categories = set(gdf2020["植生区分"])
    X_data4B_2020 = build_mesh_tensor(mesh_codes, gdf2020, category_field, categories)
    X_data4B_2022 = build_mesh_tensor(mesh_codes, gdf2022, category_field, categories)
    X_data4B_2020 = X_data4B_2020>0
    X_data4B_2022 = X_data4B_2022>0
    X_data4B = ((X_data4B_2022 + X_data4B_2020) > 0)
    del X_data4B_2022, X_data4B_2020
    N = int(len(indexlist))
    X_data = np.zeros((N,3,256,256),dtype=np.float16)
    X_data2 = np.zeros((N,5,16,16),dtype=np.float16)
    # X_data = np.zeros((N,8,256,256),dtype=np.int8)
    y_data = np.zeros((N),dtype=np.float16)
    for i in range(len(indexlist)):
        # if i % 10000 == 0:
            # print(i)
        inde = indexlist[i]
        meshid_temp = dataset["meshcode"][inde]
        A_mat, B_mat,C_mat  = get_pic_from_image("../../../../衛星写真13/" + str(dataset["meshcode"][inde]) + ".png")
        X_data[i,0,:,:] = A_mat / 256
        X_data[i,1,:,:] = B_mat / 256
        X_data[i,2,:,:] = C_mat / 256
        X_data2[i,0,:,:] = get_meshpic(meshid = meshid_temp, tiffile = 人口_4528_6242_Japan) / 1000
        X_data2[i,1,:,:] = get_meshpic(meshid = meshid_temp, tiffile = 平均標高_4528_6242_Japan) / 1000
        X_data2[i,2,:,:] = get_meshpic(meshid = meshid_temp, tiffile = 最大傾斜角度_4528_6242_Japan) / 1000
        X_data2[i,3,:,:] = get_meshpic(meshid = meshid_temp, tiffile = 最大傾斜方向_4528_6242_Japan) / 1000
        X_data2[i,4,:,:] = get_meshpic(meshid = meshid_temp, tiffile = 平均傾斜角度_人口_4528_6242_Japan) / 1000
    y_data = dataset["kuma_view"][indexlist]
    X_data3 = np.zeros((len(dataset),7),dtype=np.float16)
    X_data3[:,0] = dataset["lon"]
    X_data3[:,1] = dataset["lat"]
    X_data3[:,2] = (dataset["prefecture"] == "宮城県") * 1
    X_data3[:,3] = (dataset["prefecture"] == "山形県") * 1
    X_data3[:,4] = (dataset["prefecture"] == "岩手県") * 1
    X_data3[:,5] = (dataset["prefecture"] == "福島県") * 1
    X_data3[:,6] = (dataset["prefecture"] == "秋田県") * 1
    X_data3 = X_data3[indexlist]
    # return X_data, X_data2, X_data3, X_data4B, X_data4C, np.array(y_data)
    return restack(X_data, X_data2, X_data3, X_data4B, X_data4C), np.array(y_data)

In [11]:
def get_pic_from_image(pic_url):
    pic_temp = np.array(Image.open(pic_url).convert('RGB'))
    return pic_temp[:,:,0], pic_temp[:,:,1], pic_temp[:,:,2]

In [12]:
with tifffile.TiffFile ("../../../../熊の研究機械学習ファイル/人口_4528_6242_Japan.tif") as tif:
    人口_4528_6242_Japan = tif.asarray()
with tifffile.TiffFile ("../../../../熊の研究機械学習ファイル/平均標高_4528_6242_Japan.tif") as tif:
    平均標高_4528_6242_Japan = tif.asarray()
with tifffile.TiffFile ("../../../../熊の研究機械学習ファイル/最大傾斜角度_4528_6242_Japan.tif") as tif:
    最大傾斜角度_4528_6242_Japan = tif.asarray()
with tifffile.TiffFile ("../../../../熊の研究機械学習ファイル/最大傾斜方向_4528_6242_Japan.tif") as tif:
    最大傾斜方向_4528_6242_Japan = tif.asarray()
with tifffile.TiffFile ("../../../../熊の研究機械学習ファイル/平均傾斜角度_人口_4528_6242_Japan.tif") as tif:
    平均傾斜角度_人口_4528_6242_Japan = tif.asarray()

In [13]:
def get_meshpic(meshid, tiffile):
    x = int(str(meshid)[5]) *10*2*2 + int(str(meshid)[7]) *2*2
    y = int(str(meshid)[4]) *10*2*2 + int(str(meshid)[6]) *2*2 
    x += (int(str(meshid)[2:4])-28)*2*2*10*8
    y += (int(str(meshid)[0:2])-45)*2*2*10*8
    return tiffile[(2*2*10*8 - y - 1 - 8):(2*2*10*8 - y - 1 + 8),(x - 8):(x + 8)]

In [14]:
import pickle
def seve_metrics(name, datalist):
    with open('optina_pickeles/' + str(name) + '_train_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[0], fo)
    with open('optina_pickeles/' + str(name) + '_val_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[1], fo)
    with open('optina_pickeles/' + str(name) + '_test_loss_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[2], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[3], fo)
    with open('optina_pickeles/' + str(name) + '_val_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[4], fo)
    with open('optina_pickeles/' + str(name) + '_test_acc_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[5], fo)
 
    with open('optina_pickeles/' + str(name) + '_train_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[6], fo)
    with open('optina_pickeles/' + str(name) + '_val_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[7], fo)
    with open('optina_pickeles/' + str(name) + '_test_positive_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[8], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[9], fo)        
    with open('optina_pickeles/' + str(name) + '_val_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[10], fo)
    with open('optina_pickeles/' + str(name) + '_test_F1_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[11], fo)
        
    with open('optina_pickeles/' + str(name) + '_train_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[12], fo)        
    with open('optina_pickeles/' + str(name) + '_val_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[13], fo)
    with open('optina_pickeles/' + str(name) + '_test_AUC_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[14], fo)

    with open('optina_pickeles/' + str(name) + '_train_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[15], fo)
    with open('optina_pickeles/' + str(name) + '_valid_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[16], fo)
    with open('optina_pickeles/' + str(name) + '_test_meanscore_list.pickle', mode='wb') as fo:
        pickle.dump(datalist[17], fo)

In [15]:
import os
import joblib
import optuna
from optuna.pruners import MedianPruner

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb

In [16]:
def restack(X_data, X_data2, X_data3, X_data4B, X_data4C):
    X = np.concatenate([
    X_data.reshape((len(X_data),-1)),
    X_data2.reshape((len(X_data2),-1)),
    X_data3,
    X_data4B.reshape((len(X_data4B),-1)),
    X_data4C.reshape((len(X_data4C),-1))],-1)
    return X

In [17]:
set(dataset["prefecture"])

{'宮城県', '山形県', '岩手県', '福島県', '秋田県'}

In [18]:
# モデル保存ディレクトリ
MODEL_DIR = "optuna_models"
os.makedirs(MODEL_DIR, exist_ok=True)

# ==========================================================
# 1. 評価指標計算のヘルパー
# ==========================================================
def evaluate_binary_classifier(y_true, y_prob, threshold=0.5):
    """
    y_prob : positiveクラス (1) の予測確率ベクトル
    threshold 以上を positive とみなして評価
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    # AUCは片側クラスしかない場合にエラーになるのでガード
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    try:
        prauc = average_precision_score(y_true, y_prob)
    except ValueError:
        prauc = np.nan

    positive_rate = y_pred.mean()           # 予測ラベル=1の割合
    mean_score = y_prob.mean()              # 予測確率の平均

    metrics = {
        "auc": auc,
        "acc": acc,
        "f1": f1,
        "positive_rate": positive_rate,
        "mean_score": mean_score,
        "prauc": prauc,
    }
    return metrics

In [19]:
# ==========================================================
# 2. 汎用：Optunaで1モデルを最適化する関数
# ==========================================================
def run_optuna_for_model(
    model_name,
    prefname,
    valid_pref,
    n_trials=30,
    warm_start=False,
):
    """
    model_name : "lgbm", "rf", "dt", "logreg" のいずれか
    """
    ########## データ分割: 実験3整合の持ち回り（test=prefname / valid=valid_pref / train=残り。割合は実験3の県別件数に一致）##########
    SEED = 42
    test_index  = list(dataset.index[dataset["fold_group"] == prefname])
    valid_index = list(dataset.index[dataset["fold_group"] == valid_pref])
    train_index = list(dataset.index[(dataset["fold_group"] != prefname) & (dataset["fold_group"] != valid_pref)])
    random.seed(SEED)  # ダウンサンプリングを再現可能に
    random.shuffle(test_index)
    random.shuffle(valid_index)
    random.shuffle(train_index)

    # train の負例(kuma_view==0)を 1/10 にダウンサンプリング（正例は全て保持）
    _labels = dataset["kuma_view"]
    train_pos = [i for i in train_index if _labels[i] == 1]
    train_neg = [i for i in train_index if _labels[i] == 0]
    random.shuffle(train_neg)
    train_neg = train_neg[: len(train_neg) // 10]
    train_index = train_pos + train_neg
    random.shuffle(train_index)
    ######################################################################
        
    ######################### データローダー作成#########################
    X_train, y_train = makedata(train_index, gdf2020, gdf2022)
    X_valid, y_valid = makedata(valid_index, gdf2020, gdf2022)
    X_test, y_test = makedata(test_index, gdf2020, gdf2022)
    ######################################################################


    
    csv_path = f"optuna_results/{prefname}を予測5-{model_name}-そのまま-全県-植生あり.csv"
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    # ログ用DataFrameを初期化
    if os.path.exists(csv_path):
        final_df = pd.read_csv(csv_path, index_col=0)
    else:
        final_df = pd.DataFrame(columns=[
            "trial", "model","pref",
            # ハイパーパラメータ（モデルごとに部分的に使用）
            "n_estimators", "max_depth", "min_samples_split", "min_samples_leaf",
            "max_features", "num_leaves", "learning_rate", "reg_lambda", "reg_alpha",
            "C", "penalty", "class_weight",

            "train_y", "train_auc", "train_acc", "train_f1", "train_positive", "train_meanscore",
            "valid_y", "valid_auc", "valid_acc", "valid_f1", "valid_positive", "valid_meanscore",
            "test_y",  "test_auc",  "test_acc",  "test_f1",  "test_positive",  "test_meanscore",
        ])
        final_df.to_csv(csv_path)

    # ベストモデル格納用
    best_val_auc = -1.0
    best_model = None
    best_row = None

    def objective(trial: optuna.Trial):
        nonlocal best_val_auc, best_model, best_row, final_df


        # ------------------------------
        # モデルごとのハイパーパラメータ探索空間
        # ------------------------------
        if model_name == "lgbm":
            params = {
                "objective": "binary",
                "metric": "auc",
                "num_leaves": trial.suggest_int("num_leaves", 16, 128),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
                "random_state" : SEED ,  # 再現性のため固定
                "n_jobs": -1,
                "verbose": -1,
            }
            clf = lgb.LGBMClassifier(**params)

        elif model_name == "rf":
            params = {
                "n_estimators": trial.suggest_int("n_estimators", 100, 600),
                "max_depth": trial.suggest_int("max_depth", 3, 20),
                "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            }
            clf = RandomForestClassifier(**params, random_state=SEED)

        elif model_name == "dt":
            params = {
                "max_depth": trial.suggest_int("max_depth", 3, 20),
                "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
            }
            clf = DecisionTreeClassifier(**params, random_state=SEED)

        elif model_name == "logreg":
            # 標準化付きパイプライン
            C = trial.suggest_float("C", 1e-3, 1e3, log=True)
            penalty = "l2"   # l2固定（sagaでl1も可能だが、計算コスト増）
            class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

            base_lr = LogisticRegression(
                C=C,
                penalty=penalty,
                solver="lbfgs",
                random_state=SEED,
                max_iter=1000,
                class_weight=class_weight,
                n_jobs=-1,
            )
            clf = Pipeline([
                ("scaler", StandardScaler()),
                ("logreg", base_lr),
            ])

            # 汎用ログ用に param dict を整形
            params = {
                "C": C,
                "penalty": penalty,
                "class_weight": class_weight,
                # 他の列を埋めるためにダミー
                "n_estimators": np.nan,
                "max_depth": np.nan,
                "min_samples_split": np.nan,
                "min_samples_leaf": np.nan,
                "max_features": None,
                "num_leaves": np.nan,
                "learning_rate": np.nan,
                "reg_lambda": np.nan,
                "reg_alpha": np.nan,
            }
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        # ------------------------------
        # 学習
        # ------------------------------
        if model_name == "lgbm":
            clf.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
            )
        else:
            clf.fit(X_train, y_train)

        # --- per-trial モデル保存（モデル名＋trial番号で上書き回避）---
        joblib.dump(clf, os.path.join(MODEL_DIR, f"{model_name}_trial{trial.number}_best.pkl"))

        # ------------------------------
        # 予測確率
        # ------------------------------
        if model_name == "logreg":
            # Pipelineなので最後のステップでpredict_proba
            train_prob = clf.predict_proba(X_train)[:, 1]
            valid_prob = clf.predict_proba(X_valid)[:, 1]
            test_prob  = clf.predict_proba(X_test)[:, 1]
        else:
            train_prob = clf.predict_proba(X_train)[:, 1]
            valid_prob = clf.predict_proba(X_valid)[:, 1]
            test_prob  = clf.predict_proba(X_test)[:, 1]

        # ------------------------------
        # 各データセットで評価
        # ------------------------------
        train_metrics = evaluate_binary_classifier(y_train, train_prob)
        valid_metrics = evaluate_binary_classifier(y_valid, valid_prob)
        test_metrics  = evaluate_binary_classifier(y_test,  test_prob)

        # ------------------------------
        # CSVにログを1行追加
        # ------------------------------
        trial_id = len(final_df)

        row = {
            "trial": trial_id,
            "model": model_name,
            "pref": prefname,

            "n_estimators": params.get("n_estimators", np.nan),
            "max_depth": params.get("max_depth", np.nan),
            "min_samples_split": params.get("min_samples_split", np.nan),
            "min_samples_leaf": params.get("min_samples_leaf", np.nan),
            "max_features": params.get("max_features", None),
            "num_leaves": params.get("num_leaves", np.nan),
            "learning_rate": params.get("learning_rate", np.nan),
            "reg_lambda": params.get("reg_lambda", np.nan),
            "reg_alpha": params.get("reg_alpha", np.nan),
            "C": params.get("C", np.nan),
            "penalty": params.get("penalty", None),
            "class_weight": params.get("class_weight", None),

            "train_y": y_train.mean(),
            "train_auc": train_metrics["auc"],
            "train_prauc": train_metrics["prauc"],
            "train_acc": train_metrics["acc"],
            "train_f1":  train_metrics["f1"],
            "train_positive": train_metrics["positive_rate"],
            "train_meanscore": train_metrics["mean_score"],

            "valid_y": y_valid.mean(),
            "valid_auc": valid_metrics["auc"],
            "valid_prauc": valid_metrics["prauc"],
            "valid_acc": valid_metrics["acc"],
            "valid_f1":  valid_metrics["f1"],
            "valid_positive": valid_metrics["positive_rate"],
            "valid_meanscore": valid_metrics["mean_score"],

            "test_y": y_test.mean(),
            "test_auc": test_metrics["auc"],
            "test_prauc": test_metrics["prauc"],
            "test_acc": test_metrics["acc"],
            "test_f1":  test_metrics["f1"],
            "test_positive": test_metrics["positive_rate"],
            "test_meanscore": test_metrics["mean_score"],
        }

        final_df = pd.concat([final_df, pd.DataFrame([row])], ignore_index=True)
        final_df.to_csv(csv_path)

        # ------------------------------
        # ベストモデルの更新
        # ------------------------------
        val_auc = valid_metrics["auc"]
        if (not np.isnan(val_auc)) and (val_auc > best_val_auc):
            best_val_auc = val_auc
            best_model = clf
            best_row = row
        print("val_auc is ",val_auc)
        return val_auc

    # ------------------------------
    # Optuna Study 実行
    # ------------------------------
    study = optuna.create_study(
        study_name=f"{model_name}_fold",
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    )
    # ------------------------------
    # ウォームスタート: 既存CSVの完了済みtrialをstudyに登録し、
    # TPEサンプラーの事前情報として活用する（既存trialは再学習しない）。
    #   → 途中で止まっても、CSVに残った結果を活かして追加trialを続行できる。
    # ------------------------------
    if warm_start and len(final_df) > 0:
        _lgbm_dist = {
            "num_leaves":    optuna.distributions.IntDistribution(16, 128),
            "max_depth":     optuna.distributions.IntDistribution(3, 10),
            "learning_rate": optuna.distributions.FloatDistribution(1e-3, 1e-1, log=True),
            "n_estimators":  optuna.distributions.IntDistribution(100, 600),
            "reg_lambda":    optuna.distributions.FloatDistribution(1e-3, 10.0, log=True),
        }
        added = 0
        for _, prow in final_df.iterrows():
            if pd.isna(prow.get("valid_auc")):
                continue
            try:
                if model_name == "lgbm":
                    params_done = {
                        "num_leaves":    int(prow["num_leaves"]),
                        "max_depth":     int(prow["max_depth"]),
                        "learning_rate": float(prow["learning_rate"]),
                        "n_estimators":  int(prow["n_estimators"]),
                        "reg_lambda":    float(prow["reg_lambda"]),
                    }
                    dist_done = _lgbm_dist
                else:
                    print(f"warm_start は現状 lgbm のみ対応のためスキップ（model={model_name}）")
                    break
                study.add_trial(
                    optuna.trial.create_trial(
                        params=params_done,
                        distributions=dist_done,
                        value=float(prow["valid_auc"]),
                    )
                )
                added += 1
            except Exception as e:
                print("warm-start 登録をスキップ:", e)
        print(f"[warm-start] 既存 {added} trial を study に登録（再学習なし）。"
              f"これらを事前情報として追加 {n_trials} trial を実行します。")

    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, n_jobs=1)

    # ------------------------------
    # ベストモデルを保存
    # ------------------------------
    if best_model is not None:
        model_path = os.path.join(MODEL_DIR, f"{model_name}_best.pkl")
        joblib.dump(best_model, model_path)
        print(f"[{model_name}] best model saved to: {model_path}")
        print(f"[{model_name}] best valid AUC: {best_val_auc:.4f}")
    else:
        print(f"[{model_name}] No valid model found.")

    return study, best_model, best_row

In [ ]:
# LightGBM 持ち回り5fold（test/valid を fold_group で回す。割合は実験3の県別件数に一致）
FOLDS = [("宮城県","山形県"),("山形県","岩手県"),("岩手県","福島県"),("福島県","秋田県"),("秋田県","宮城県")]
for TEST_GRP, VALID_GRP in FOLDS:
    print(f"=== LGBM fold: test={TEST_GRP}, valid={VALID_GRP} ===")
    study_lgbm, best_lgbm, row_lgbm = run_optuna_for_model(
        model_name="lgbm",
        prefname=TEST_GRP,
        valid_pref=VALID_GRP,
        n_trials=5,
    )


[I 2026-06-08 00:45:16,355] A new study created in memory with name: lgbm_fold


In [ ]:
# 【5fold一括実行では無効化】warm-start による追加trialは行わない（trial数は各fold 5 に固定）
print("warm-start セルは 5fold 一括実行のためスキップ")


In [ ]:
# 【5fold一括実行では無効化】保存済みモデルからのPRAUC再算出(単一分割・モデル横断)はスキップ。
# PR-AUC は各fold本体CSVの *_prauc 列に記録済み。
print("PRAUC診断セルは 5fold 一括実行のためスキップ")
